# Notebook Synchronization with Jupytext

## Why Use Jupytext?
* **Version Control (Git) Friendly:** Standard `.ipynb` files contain heavy JSON metadata and outputs, making Git diffs unreadable. Re-saving them as pure Python scripts (`.py`) allows clean tracking of code changes.
* **IDE Compatibility:** The `percent` format structure (`# %%`) lets you easily open, edit, and run these notebooks in powerful Python IDEs like VS Code or PyCharm.
* **Two-Way Synchronization:** Jupytext links the `.ipynb` file and the `.py` script. Changes made in the JupyterLab UI will automatically update the Python file, and vice versa.


In [1]:
!pip install -q jupytext --upgrade

In [2]:
import os
import glob
import nbformat
import warnings

# Suppress warnings in the current output window for clean logging
warnings.filterwarnings("ignore", category=UserWarning, module="nbformat")

print("Step 1: Normalizing notebooks to fix MissingIDFieldWarning...")

# Find all .ipynb files in the current directory
notebook_files = glob.glob("*.ipynb")

for nb_file in notebook_files:
    try:
        # Read the notebook using nbformat
        with open(nb_file, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)
        
        # This adds missing 'id' fields to all cells transparently
        # keeping the metadata compatible with new nbformat standards
        nbformat.validate(nb) 
        
        # Save the updated notebook back to disk
        with open(nb_file, "w", encoding="utf-8") as f:
            nbformat.write(nb, f)
            
    except Exception as e:
        print(f" Could not automatically normalize {nb_file}: {e}")

print(" Notebooks successfully normalized.")

print("\nStep 2: Running Jupytext synchronization...")

# 1. Define output directory for synchronized scripts
output_dir = "jupytext_py_percent"
os.makedirs(output_dir, exist_ok=True)

# 2. Configure Jupytext pair formatting
!jupytext --set-formats "ipynb,jupytext_py_percent//py:percent" *.ipynb

# 3. Perform two-way sync
!jupytext --sync *.ipynb

print(f"\n Success! All synchronized .py files are stored in: {output_dir}")


Step 1: Normalizing notebooks to fix MissingIDFieldWarning...
 Notebooks successfully normalized.

Step 2: Running Jupytext synchronization...
[jupytext] Reading 00_introduction.ipynb in format ipynb
[jupytext] Updating notebook metadata with '{"jupytext": {"formats": "ipynb,jupytext_py_percent//py:percent"}}'
[jupytext] Unchanged 00_introduction.ipynb
[jupytext] Updating jupytext_py_percent/00_introduction.py
[jupytext] Reading 01_data_loading.ipynb in format ipynb
[jupytext] Updating notebook metadata with '{"jupytext": {"formats": "ipynb,jupytext_py_percent//py:percent"}}'
[jupytext] Unchanged 01_data_loading.ipynb
[jupytext] Updating jupytext_py_percent/01_data_loading.py
[jupytext] Reading 02_eda_basic.ipynb in format ipynb
[jupytext] Updating notebook metadata with '{"jupytext": {"formats": "ipynb,jupytext_py_percent//py:percent"}}'
[jupytext] Unchanged 02_eda_basic.ipynb
[jupytext] Updating jupytext_py_percent/02_eda_basic.py
[jupytext] Reading 03_eda_advanced.ipynb in format ip

## How to use this setup:
1. **Isolated Management:** Your main directory stays clean and clutter-free, containing only your active `.ipynb` files.
2. **IDE Integration:** You can safely open any file inside `jupytext_py_percent/` using an external IDE (like **VS Code** or **PyCharm**). 
3. **Automated Sync:** Every time you run the script above, Jupytext will look at the timestamps. 
   * If you edited the `.ipynb` file in JupyterLab, it will update the corresponding `.py` script.
   * If you edited the `.py` script in VS Code, it will port those changes back into your `.ipynb` notebook.
